# 11 — Dynamic tool spawning

**The missing rung on the autonomy ladder.**

Today the framework spawns *agents* at runtime via `AgentFactory.spawn(AgentSpec)`. But agents could only use tools that already existed in `ToolRegistry`. When an LLM emitted an `AgentSpec` requesting a brand-new capability, the agent silently spawned without it.

This notebook demonstrates the closing of that loop:

1. **Sandbox standalone** — validate + execute small Python implementations safely.
2. **`DynamicToolFactory`** — turn a `GeneratedToolSpec` (with `implementation: str`) into a runnable `pydantic_ai.Tool`.
3. **`AgentFactory` dispatch** — an `AgentSpec` whose `tools` list mixes pre-registered `ToolSpec` and freshly-generated `GeneratedToolSpec`; both resolve correctly.
4. **`BaseAgent.add_tool`** — runtime tool assignment to an existing agent (the cache invalidates so the rebuilt internal `pydantic_ai.Agent` sees the new tool).
5. **End-to-end** — a real LLM agent that has a runtime-spawned tool in its kit and uses it.

Cost: ~\$0.50 (one short LLM run in the final cell).


## 1. Sandbox standalone

Two backends ship: `InProcessSandbox` (Tier 0, requires `unsafe=True`) and `SubprocessSandbox` (Tier 1, the production default). Both honor the same Protocol — validate then execute.

In [1]:
import asyncio
from orqest.sandbox import InProcessSandbox, SubprocessSandbox, ValidationError

# Tier 0: in-process exec with AST-level static restriction.
# Refuses to construct without unsafe=True.
inproc = InProcessSandbox(unsafe=True)
result = await inproc.execute(
    "return args['x'] + args['y']",
    args={'x': 6, 'y': 7},
    allowed_imports=set(),
)
print(f"in-process: success={result.success} output={result.output} dur={result.duration_ms:.2f}ms")

# Tier 1: subprocess + RLIMIT_AS + RLIMIT_CPU + outer asyncio.wait_for timeout.
sub = SubprocessSandbox()
result = await sub.execute(
    "import re; return re.findall(r'\\d+', args['text'])",
    args={'text': 'a1 b22 c333'},
    allowed_imports={'re'},
    timeout_s=2.0,
)
print(f"subprocess: success={result.success} output={result.output} dur={result.duration_ms:.0f}ms")


in-process: success=True output=13 dur=0.10ms
subprocess: success=True output=['1', '22', '333'] dur=67ms


### Validate-time rejection — default-deny imports

The static AST walk rejects imports not in `allowed_imports`, calls to forbidden names (`eval`, `exec`, `__import__`), and dunder attribute access (`__class__`, `__subclasses__`).

In [2]:
try:
    await sub.validate('import os; return os.getcwd()', allowed_imports=set())
except ValidationError as exc:
    print(f'rejected: {exc}')

try:
    await sub.validate('return eval("1+1")', allowed_imports=set())
except ValidationError as exc:
    print(f'rejected: {exc}')


rejected: line 1: import 'os' not in allowed_imports (empty) (Import)
rejected: line 1: call to forbidden name 'eval' (Call); line 1: reference to forbidden name 'eval' (Name)


### Execute-time timeout — kills the subprocess

In [3]:
result = await sub.execute(
    'while True: pass',
    args={},
    allowed_imports=set(),
    timeout_s=1.0,
)
print(f"timeout: success={result.success} error={(result.error or '')[:80]}")


timeout: success=False error=sandbox execution timed out after 1.00s


## 2. `DynamicToolFactory` — spec to live `pydantic_ai.Tool`

`DynamicToolFactory.spawn(GeneratedToolSpec)` validates the implementation through the configured sandbox, then returns a `pydantic_ai.Tool` whose body delegates to `sandbox.execute()` at invocation time.

In [4]:
from orqest.autonomy import DynamicToolFactory, GeneratedToolSpec

factory = DynamicToolFactory(SubprocessSandbox())

doi_spec = GeneratedToolSpec(
    name='extract_dois',
    description='Extract DOIs from a text blob.',
    parameters={'text': {'type': 'string'}},
    implementation=(
        "import re\n"
        "matches = re.findall(r'10\\.\\d{4,}/[\\w.\\-/]+', args['text'])\n"
        "return {'dois': matches, 'count': len(matches)}\n"
    ),
    allowed_imports={'re'},
)

doi_tool = await factory.spawn(doi_spec)
print(f'spawned tool: {doi_tool.name} - {doi_tool.description}')

# Invoke directly via the runner closure
sample = 'See Smith (2024) at 10.1234/foo, also Jones (2025) at 10.5678/bar/baz.'
out = await doi_tool.function(text=sample)
print(f'invoked: {out}')


spawned tool: extract_dois - Extract DOIs from a text blob.
invoked: {'dois': ['10.1234/foo', '10.5678/bar/baz.'], 'count': 2}


## 3. `AgentFactory` dispatch — mixed registered + generated tools

An `AgentSpec` whose `tools` list contains both kinds resolves through the registry (for `ToolSpec`) and through the `DynamicToolFactory` (for `GeneratedToolSpec`).

In [5]:
from pydantic_ai import Tool
from pydantic_ai.models.test import TestModel
from orqest.autonomy import AgentFactory, AgentSpec, ToolSpec, ToolRegistry


# Pre-register a tool the conventional way
async def search(q: str) -> str:
    """Search the web."""
    return f'searched: {q}'


registry = ToolRegistry()
registry.register(Tool(search, name='search'))

agent_factory = AgentFactory(registry=registry, tool_factory=factory)

spec = AgentSpec(
    name='researcher',
    system_prompt='You research topics and extract DOIs.',
    output_schema={'type': 'object', 'properties': {'answer': {'type': 'string'}}, 'required': ['answer']},
    tools=[
        ToolSpec(name='search', description='Search the web'),         # registered
        GeneratedToolSpec(                                              # generated
            name='extract_dois',
            description='Extract DOIs from text.',
            parameters={'text': {'type': 'string'}},
            implementation=doi_spec.implementation,
            allowed_imports={'re'},
        ),
    ],
)

agent = agent_factory.spawn(spec, model=TestModel())
print(f'agent tools: {sorted(t.name for t in agent.tools)}')


agent tools: ['extract_dois', 'search']


## 4. `BaseAgent.add_tool` — runtime tool assignment to existing agents

`add_tool()` appends to `self.tools` and invalidates the cached internal `pydantic_ai.Agent` so the next access rebuilds with the new tool list. Idempotent for tools sharing a `name` (last-write-wins).

In [6]:
# Spawn a fresh agent with NO tools
bare_agent = agent_factory.spawn(
    AgentSpec(
        name='bare',
        system_prompt='you do nothing initially',
        output_schema={'type': 'object', 'properties': {'answer': {'type': 'string'}}, 'required': ['answer']},
        tools=[],
    ),
    model=TestModel(),
)
print(f'before: tools={[t.name for t in bare_agent.tools]} _agent cached? {bare_agent._agent is not None}')

# Force the cache to populate
_ = bare_agent.agent
print(f'after access: _agent cached? {bare_agent._agent is not None}')

# Hand the agent the freshly-spawned DOI tool
bare_agent.add_tool(doi_tool)
print(f'after add_tool: tools={[t.name for t in bare_agent.tools]} _agent cached? {bare_agent._agent is not None}')

# Next access rebuilds with the new tool
_ = bare_agent.agent
print(f'after rebuild: _agent rebuilt with {[t.name for t in bare_agent.tools]}')


before: tools=[] _agent cached? False
after access: _agent cached? True
after add_tool: tools=['extract_dois'] _agent cached? False
after rebuild: _agent rebuilt with ['extract_dois']


## 5. End-to-end — real LLM agent uses a runtime-spawned tool

A real `BaseAgent` (with the project's default model) gets the spawned `extract_dois` tool added at runtime, then runs against a prompt that exercises it.

In [7]:
from pydantic import BaseModel, Field
from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()


class DoiAnswer(BaseModel):
    answer: str = Field(description='Comma-separated list of DOIs found, or "none".')


class DoiAgent(BaseAgent[GlobalState, DoiAnswer]):
    async def _run_implementation(self, state, **kwargs):
        latest = state.get_latest_message('user') or ''
        result = await self.call_model(latest, state)
        return result.output


# Spawn agent without tools, then add the freshly-spawned DOI tool at runtime
real_agent = DoiAgent(
    agent_name='doi_finder',
    system_prompt=(
        'You find DOIs in user-supplied text. Use the extract_dois tool '
        'to scan the text, then report the DOIs found in your answer.'
    ),
    output_type=DoiAnswer,
    model=config.llm_model,
    api_key=config.llm_api_key,
)

real_agent.add_tool(doi_tool)
print(f'real_agent tools: {[t.name for t in real_agent.tools]}')

state = GlobalState()
state.add_message('user', f'Find the DOIs in this text:\n\n{sample}')
result = await real_agent.run(state)
print(f'final answer: {result.answer}')


real_agent tools: ['extract_dois']


final answer: 10.1234/foo, 10.5678/bar/baz.


## 6. Bus events — observability

Subscribe to `tool.spawned`, `tool.invocation_completed`, and `sandbox.validation_rejected` to surface the tool lifecycle.

In [8]:
from orqest.observability.events import EventBus

bus = EventBus()
events = []
for et in ('tool.spawned', 'tool.invocation_completed', 'tool.invocation_failed', 'sandbox.validation_rejected'):
    bus.subscribe(et, events.append)

bus_factory = DynamicToolFactory(SubprocessSandbox(), bus=bus)

# Successful spawn + invoke
spec_ok = GeneratedToolSpec(
    name='reverse',
    description='Reverse a string.',
    parameters={'text': {'type': 'string'}},
    implementation='return args["text"][::-1]',
    allowed_imports=set(),
)
tool_ok = await bus_factory.spawn(spec_ok)
await tool_ok.function(text='hello')

# Rejected spawn (disallowed import)
try:
    await bus_factory.spawn(GeneratedToolSpec(
        name='bad',
        description='x',
        parameters={},
        implementation='import os; return os.getcwd()',
        allowed_imports=set(),
    ))
except ValidationError:
    pass

await asyncio.sleep(0.05)  # let bus tasks flush
for e in events:
    print(f'  {e.event_type}: {e.data}')


  tool.spawned: {'tool_name': 'reverse'}
  tool.invocation_completed: {'tool_name': 'reverse', 'duration_ms': 168.75600399998802}
  sandbox.validation_rejected: {'tool_name': 'bad', 'reason': "line 1: import 'os' not in allowed_imports (empty) (Import)"}


## What's next

- **W3.J — Procedural-memory persistence for spawned tools.** A spawned tool that proves itself across multiple invocations becomes a `Skill` in `LocalMemoryStore`; future runs recall + re-spawn it without re-asking the LLM. The bus events fire today; W3.J wires the persistence side.
- **W3.K — `SubprocessPoolSandbox`.** Reuse pre-warmed Python subprocesses to amortize the per-invocation startup cost (~50ms today).
- **W3.L — `E2BSandbox` / `DockerSandbox`.** Real isolation backends (network firewall, filesystem isolation) for higher-trust scenarios.
- **W3.C revisited — ADAS sandboxed codegen.** With the sandbox now in core, `MetaAgentSearch` can be extended to allow the meta agent to emit raw Python `forward()` for cases where compositions of registered primitives are too constrained.

For the design rationale, see `docs/concepts/sandbox.md` and the "Dynamic tool spawning" section of `docs/concepts/autonomy.md`.
